# 🎬 Maddələr Əsaslı Birgə Filtrləmə ilə Oxşar Filmlərin Tapılması
### (Finding Similar Movies with Item-Based Collaborative Filtering)

Bu notebookda müasir tövsiyə sistemlərinin (Recommendation Systems) təməl prinsiplərindən biri olan **Maddələr Əsaslı Birgə Filtrləmə (Item-Based Collaborative Filtering)** yanaşmasını addım-addım tətbiq edəcəyik.

#### 📌 Əsas Məqsəd və İdeya:
- İki filmin bir-birinə bənzər olub-olmadığını onların janrına, rejissoruna və ya mətn təsvirinə görə deyil, **istifadəçilərin onlara verdiyi qiymətləndirmələrin (ratings) oxşarlığına** əsasən təyin edəcəyik.
- Əgər bir çox istifadəçi "A" filmini bəyənibsə və eyni zamanda "B" filminə də yüksək bal veribsə, sistem "A" və "B" filmlərini oxşar hesab edir.
- Nümunə olaraq əfsanəvi **"Star Wars (1977)"** filmini seçəcək və ona ən çox bənzəyən digər filmləri statistik korrelyasiya vasitəsilə aşkarlayacağıq.

## 📂 1. Məlumat Dəstinin (MovieLens 100K) Yüklənməsi və Birləşdirilməsi

GroupLens tədqiqat qrupunun məşhur **MovieLens 100K** verilənlər bazasından istifadə edirik:
- ml-100k/u.data: 943 istifadəçinin 1682 filmə verdiyi 100,000 qiymətləndirmə (sütunlar: user_id, movie_id, 
ating, 	imestamp).
- ml-100k/u.item: Filmlərin ID-si, adı və buraxılış ili (sütunlar: movie_id, 	itle, ...).

> 💡 **Praktiki Qeyd:** Real istehsalat mühitində (production) yaddaşa qənaət və hesablama sürəti üçün bütün əməliyyatlar yalnız movie_id üzərində aparılır, adlar isə yalnız istifadəçiyə göstərilən interfeys qatında əlavə edilir. Lakin dərsin vizual və anlaşıqlı olması üçün 
atings və movies cədvəllərini movie_id əsasında **pd.merge** ilə birləşdirəcəyik.

In [ ]:
import pandas as pd
import numpy as np

# u.data faylından ilk 3 sütunu oxuyuruq: istifadəçi ID-si, film ID-si və qiymət (1-5 xal)
r_cols = ['user_id', 'movie_id', 'rating']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols, usecols=range(3), encoding="ISO-8859-1")

ratings

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3
...,...,...,...
99998,880,476,3
99999,716,204,5
100000,276,1090,1
100001,13,225,2


In [3]:
# u.item faylından film ID-sini və film adını oxuyuruq (faylda sütun ayırıcısı '|' simvoludur)
m_cols = ['movie_id', 'title']
movies = pd.read_csv('ml-100k/u.item', sep='|', names=m_cols, usecols=range(2), encoding="ISO-8859-1")

movies

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
...,...,...
1677,1678,Mat' i syn (1997)
1678,1679,B. Monkey (1998)
1679,1680,Sliding Doors (1998)
1680,1681,You So Crazy (1994)


In [4]:
# 'movie_id' ortaq sütunu üzrə hər iki cədvəli birləşdiririk (Merge)
ratings = pd.merge(movies, ratings)

ratings

,movie_id,title,user_id,rating
0,1,Toy Story (1995),308,4
1,1,Toy Story (1995),287,5
2,1,Toy Story (1995),148,4
3,1,Toy Story (1995),280,4
4,1,Toy Story (1995),66,3
...,...,...,...,...
99998,1678,Mat' i syn (1997),863,1
99999,1679,B. Monkey (1998),863,3
100000,1680,Sliding Doors (1998),863,2
100001,1681,You So Crazy (1994),896,3


## 🗂️ 2. İstifadəçi-Film Qiymətləndirmə Matrisinin Qurulması (pivot_table)

Pandas-ın **pivot_table** funksiyası vasitəsilə 2-ölçülü seyrək matris (Sparse Matrix) formalaşdırırıq:
- **Sətirlər (Index):** user_id (hər bir fərdi istifadəçi).
- **Sütunlar (Columns):** 	itle (bütün 1682 filmin adı).
- **Xanaların dəyəri (Values):** 
ating (istifadəçinin həmin filmə verdiyi 1-5 arası bal).

> ⚠️ **Seyrək Matris (Sparse Matrix) Qanunauyğunluğu:**  
> Xanaların əksəriyyətində **NaN** (Not a Number — boş xana) görəcəksiniz. Bu tamamilə təbii haldır, çünki heç bir istifadəçi bazadakı 1682 filmin hamısını izləməyib. Tövsiyə sistemlərində bu vəziyyət **Seyrəklik (Sparsity)** adlanır.

In [5]:
# İstifadəçi-Film matrisini qururuq:
movieRatings = ratings.pivot_table(index=['user_id'], columns=['title'], values='rating')
movieRatings

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
941,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 🎯 3. Hədəf Filmin Seçilməsi: "Star Wars (1977)"

İndi matrisdən konkret bir filmin — məşhur **"Star Wars (1977)"** filminin bütün istifadəçilər üzrə qiymətləndirmə vektorunu (Series) ayırırıq.

Bu vektor hər bir istifadəçinin Star Wars-a neçə bal verdiyini göstərir (əgər istifadəçi bu filmi izləməyibsə, qiymət NaN olacaq).

In [6]:
# 'Star Wars (1977)' sütununu ayrıca Series kimi çıxarırıq:
starWarsRatings = movieRatings['Star Wars (1977)']
starWarsRatings.head()

user_id
0    5.0
1    5.0
2    5.0
3    NaN
4    5.0
Name: Star Wars (1977), dtype: float64

## 📐 4. Pirson Korrelyasiyası ilə Oxşarlığın Hesablanması (corrwith)

Pandas-ın **corrwith** metodu Star Wars filminin qiymətləndirmə vektoru ilə bazadakı digər bütün filmlərin vektorları arasında cüt-cüt **Pirson Korrelyasiya Əmsalını (Pearson Correlation)** hesablayır:

\rho = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}

- Əmsal **+1.0** olduqda: Filmləri izləyən istifadəçilərin zövqü mükəmməl dərəcədə eynidir (hər iki filmə eyni səviyyədə yüksək və ya aşağı bal veriblər).
- Əmsal **0.0** olduqda: Filmlər arasında heç bir xətti statistik əlaqə yoxdur.
- Əmsal **-1.0** olduqda: Əks zövq (birini bəyənənlər digərinə nifrət edib).

Hesablamadan sonra hər iki filmi eyni vaxtda qiymətləndirməyən halları (NaN) **dropna()** ilə təmizləyirik.

In [7]:
# Star Wars vektoru ilə digər bütün filmlər arasındakı cüt-cüt korrelyasiyanı tapırıq:
similarMovies = movieRatings.corrwith(starWarsRatings)
similarMovies = similarMovies.dropna()
similarMovies

c:\Users\tarlan.cabiyev\AppData\Local\anaconda3\envs\ayna\lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\tarlan.cabiyev\AppData\Local\anaconda3\envs\ayna\lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)


title
'Til There Was You (1997)                0.872872
1-900 (1994)                            -0.645497
101 Dalmatians (1996)                    0.211132
12 Angry Men (1957)                      0.184289
187 (1997)                               0.027398
                                           ...   
Young Guns (1988)                        0.186377
Young Guns II (1990)                     0.228615
Young Poisoner's Handbook, The (1995)   -0.007374
Zeus and Roxanne (1997)                  0.818182
unknown                                  0.723123
Length: 1410, dtype: float64

In [17]:
# Nəticəni cədvəl kimi saxlamaq və emal etmək üçün DataFrame formatına keçiririk:
df = pd.DataFrame(similarMovies)
df

,0
title,
'Til There Was You (1997),0.872872
1-900 (1994),-0.645497
101 Dalmatians (1996),0.211132
12 Angry Men (1957),0.184289
187 (1997),0.027398
...,...
Young Guns (1988),0.186377
Young Guns II (1990),0.228615
"Young Poisoner's Handbook, The (1995)",-0.007374


## ⚠️ 5. Təhlil və Gözlənilməz Nəticə: Yalançı Korrelyasiya (Spurious Correlation)

Gəlin nəticələri oxşarlıq xalına görə azalan sıra ilə düzək və Star Wars-a ən çox bənzəyən filmlərə baxaq.

Məntiqən siyahının başında Star Wars-un digər hissələri və ya digər elmi-fantastik şedevrlər olmalıdır... Amma gəlin nəticəyə birlikdə baxaq!

In [18]:
# Oxşarlıq xalına görə ən yüksəkdən aşağıya sıralayırıq:
similarMovies.sort_values(ascending=False)

title
Commandments (1997)           1.0
Cosi (1996)                   1.0
No Escape (1994)              1.0
Stripes (1981)                1.0
Man of the Year (1995)        1.0
                             ... 
For Ever Mozart (1996)       -1.0
Frankie Starlight (1995)     -1.0
I Like It Like That (1994)   -1.0
American Dream (1990)        -1.0
Theodore Rex (1995)          -1.0
Length: 1410, dtype: float64

### 🛑 Nə Baş Verdi? Niyə Belə Naməlum Filmlər Çıxdı?

Nəticədə *'Til There Was You (1997)*, *Substance of Fire, The (1996)*, *Quiet Room, The (1996)* kimi filmlərin oxşarlığı **1.0 (100%)** çıxır! Bu nəticə real həyat məntiqinə tamamilə ziddir.

#### 🔍 Səbəb: "Yalançı Korrelyasiya" (Spurious Correlation)
- Bu filmlər cəmi 1 və ya 2 istifadəçi tərəfindən izlənilmişdir.
- Təsəvvür edin: təsadüfən cəmi 1 istifadəçi həm *Star Wars*-a 5 xal verib, həm də həmin bu naməlum filmə 5 xal verib.
- İki nöqtə arasında riyazi korrelyasiya hesablandıqda xətt mükəmməl düz alınır və nəticə **1.0** çıxır!
- Lakin cəmi 1 nəfərin rəyinə əsaslanaraq qlobal tövsiyə vermək tamamilə **statistik xətadır**.

#### 💡 Həll Yolu (Threshold Filtering):
Hər bir filmin **ümumi neçə nəfər tərəfindən qiymətləndirildiyini (size)** və **orta balını (mean)** hesablamaq, yalnız kifayət qədər böyük auditoriya toplamış filmləri saxlamaq!

In [ ]:
# Hər bir film üzrə qiymətləndirmə sayını (np.size) və orta balı (np.mean) qruplaşdırıb hesablayırıq:
movieStats = ratings.groupby('title').agg({'rating': [np.size, np.mean]})
movieStats.head()

rating          
                            size      mean
title                                     
'Til There Was You (1997)      9  2.333333
1-900 (1994)                   5  2.600000
101 Dalmatians (1996)        109  2.908257
12 Angry Men (1957)          125  4.344000
187 (1997)                    41  3.024390

## 🧹 6. İzləmə Sayına Görə Filtrləmə (Hədd: Ən Azı 100 Rəy)

Statistik etibarlılığı təmin etmək üçün **ən azı 100 nəfər tərəfindən qiymətləndirilmiş** filmləri seçirik (size >= 100).

Gəlin bu meyara cavab verən və ən yüksək orta bala sahib olan ilk 15 filmə nəzər salaq:

In [20]:
# Ən azı 100 istifadəçi tərəfindən qiymətləndirilmiş filmləri seçirik və orta bala görə sıralayırıq:
popularMovies = movieStats['rating']['size'] >= 100
movieStats[popularMovies].sort_values([('rating', 'mean')], ascending=False)[:15]

rating          
                                         size      mean
title                                                  
Close Shave, A (1995)                     112  4.491071
Schindler's List (1993)                   298  4.466443
Wrong Trousers, The (1993)                118  4.466102
Casablanca (1942)                         243  4.456790
Shawshank Redemption, The (1994)          283  4.445230
Rear Window (1954)                        209  4.387560
Usual Suspects, The (1995)                267  4.385768
Star Wars (1977)                          584  4.359589
12 Angry Men (1957)                       125  4.344000
Citizen Kane (1941)                       198  4.292929
To Kill a Mockingbird (1962)              219  4.292237
One Flew Over the Cuckoo's Nest (1975)    264  4.291667
Silence of the Lambs, The (1991)          390  4.289744
North by Northwest (1959)                 179  4.284916
Godfather, The (1972)                     413  4.283293

## 🔗 7. Korrelyasiya Nəticələri ilə Statistik Göstəricilərin Birləşdirilməsi

İndi yalnız 100-dən çox rəy toplamış populyar filmlərin cədvəlini əvvəlcə hesabladığımız similarMovies (Star Wars ilə korrelyasiya) cədvəli ilə **join** edirik.

> 🛠️ **Pandas Uyğunluğu (MultiIndex Sütunların Düzləşdirilməsi):**  
> movieStats cədvəlində sütunlar iki səviyyəlidir (MultiIndex: ('rating', 'size') və ('rating', 'mean')). Yeni Pandas versiyalarında fərqli səviyyəli sütunları birləşdirərkən xəta yaranmaması üçün əvvəlcə sütun adlarını 
ating|size və 
ating|mean kimi tək səviyyəli (flat) formata gətiririk.

In [21]:
# Filtrlənmiş populyar filmləri ayrıca dəyişənə mənimsədirik:
mappedColumnsMoviestat = movieStats[popularMovies]
mappedColumnsMoviestat

rating          
                                               size      mean
title                                                        
101 Dalmatians (1996)                           109  2.908257
12 Angry Men (1957)                             125  4.344000
2001: A Space Odyssey (1968)                    259  3.969112
Absolute Power (1997)                           127  3.370079
Abyss, The (1989)                               151  3.589404
...                                             ...       ...
Willy Wonka and the Chocolate Factory (1971)    326  3.631902
Wizard of Oz, The (1939)                        246  4.077236
Wrong Trousers, The (1993)                      118  4.466102
Young Frankenstein (1974)                       200  3.945000
Young Guns (1988)                               101  3.207921

[338 rows x 2 columns]

In [22]:
# Mövcud MultiIndex sütun strukturunu yoxlayırıq:
mappedColumnsMoviestat.columns

MultiIndex([('rating', 'size'),
            ('rating', 'mean')],
           )

In [24]:
# Sütun adlarını düzləşdiririk (məsələn: 'rating|size', 'rating|mean'):
mappedColumnsMoviestat.columns = [f'{i}|{j}' if j != '' else f'{i}' for i, j in mappedColumnsMoviestat.columns]
mappedColumnsMoviestat

,rating|size,rating|mean
title,,
101 Dalmatians (1996),109,2.908257
12 Angry Men (1957),125,4.344000
2001: A Space Odyssey (1968),259,3.969112
Absolute Power (1997),127,3.370079
"Abyss, The (1989)",151,3.589404
...,...,...
Willy Wonka and the Chocolate Factory (1971),326,3.631902
"Wizard of Oz, The (1939)",246,4.077236
"Wrong Trousers, The (1993)",118,4.466102


In [26]:
# Düzləşdirilmiş statistik göstəriciləri Star Wars oxşarlıq xalları ilə birləşdiririk:
df = mappedColumnsMoviestat.join(pd.DataFrame(similarMovies, columns=['similarity']))
df

,rating|size,rating|mean,similarity
title,,,
101 Dalmatians (1996),109,2.908257,0.211132
12 Angry Men (1957),125,4.344000,0.184289
2001: A Space Odyssey (1968),259,3.969112,0.230884
Absolute Power (1997),127,3.370079,0.085440
"Abyss, The (1989)",151,3.589404,0.203709
...,...,...,...
Willy Wonka and the Chocolate Factory (1971),326,3.631902,0.221902
"Wizard of Oz, The (1939)",246,4.077236,0.266335
"Wrong Trousers, The (1993)",118,4.466102,0.216204


## 🏆 8. Son Nəticə: Star Wars (1977) üçün Ən Uyğun Tövsiyələr!

İndi birləşdirilmiş cədvəli similarity sütununa görə azalan sıra ilə düzək və Star Wars-a ən oxşar ilk 15 filmi görək:

In [27]:
# Oxşarlıq əmsalına görə ən yüksək 15 filmi sıralayırıq:
df.sort_values(['similarity'], ascending=False)[:15]

,rating|size,rating|mean,similarity
title,,,
Star Wars (1977),584,4.359589,1.000000
"Empire Strikes Back, The (1980)",368,4.206522,0.748353
Return of the Jedi (1983),507,4.007890,0.672556
Raiders of the Lost Ark (1981),420,4.252381,0.536117
Austin Powers: International Man of Mystery (1997),130,3.246154,0.377433
"Sting, The (1973)",241,4.058091,0.367538
Indiana Jones and the Last Crusade (1989),331,3.930514,0.350107
Pinocchio (1940),101,3.673267,0.347868
"Frighteners, The (1996)",115,3.234783,0.332729


### 🌟 Nəticələrin Dərin Təhlili və Qiymətləndirilməsi:

Nəticələrə diqqət yetirin — onlar sadəcə heyrətamiz dərəcədə məntiqlidir!
1. **Star Wars (1977)**: similarity = 1.0 (hər bir film təbii ki, özü ilə 100% korrelyasiyadadır).
2. **Empire Strikes Back, The (1980)**: similarity = 0.748 — Star Wars silsiləsinin 2-ci filmi (Epizod V)!
3. **Return of the Jedi (1983)**: similarity = 0.673 — Star Wars klassik trilogiyasının 3-cü filmi (Epizod VI)!
4. **Raiders of the Lost Ark (1981)**: similarity = 0.536 — Eyni zamanda Corc Lukasın yaratdığı və baş rolda Star Wars-un ulduzu Harrison Fordun (Han Solo) çəkildiyi İndiana Cons filmi!
5. **Austin Powers (1997)**, **Sting, The (1973)**, **Indiana Jones and the Last Crusade (1989)** və s. — oxşar auditoriya kütləsinin sevdiyi filmlər.

> 🎯 **Nəticə:** Heç bir janr mətni oxumadan, aktyor və rejissor siyahısına baxmadan, **yalnız istifadəçilərin qiymətləndirmə davranışlarına və statistik korrelyasiyaya əsaslanaraq** sistem Star Wars silsiləsinin digər filmlərini dəqiqliklə tapdı!

In [ ]:
# Bonus Təmizləmə: Filmin özünü ('Star Wars (1977)') siyahıdan xaric edərək
# istifadəçiyə təqdim ediləcək Ən Yaxşı 10 Tövsiyəni (Top-10 Recommendations) əldə edirik:
top10_recommendations = df.drop('Star Wars (1977)').sort_values(['similarity'], ascending=False).head(10)
top10_recommendations